# 04 — Prontuário estruturado: SQLite, consultas e minimização de dados

**Tech Challenge Fase 3 — MedFlow AI**

O enunciado exige que o assistente "realize consultas em base de dados estruturadas (como prontuários
e registros)" e "contextualize as respostas com informações atualizadas do paciente". Este notebook
mostra a base, as consultas e — o ponto crítico — **o que sai e o que não sai** da base rumo à LLM.

In [ ]:
import sys, pathlib
RAIZ = pathlib.Path.cwd()
while not (RAIZ / "src" / "medflow_ai").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
print("Raiz do projeto:", RAIZ)

## 1. Por que dados sintéticos gerados, e não Synthea baixado?

O esquema é **idêntico ao export CSV do Synthea** (`patients`, `conditions`, `observations`,
`medications`, `procedures`, `encounters`), e existe um ingestor pronto para um export real
(`ingest_synthea_csv`). O gerador próprio foi mantido como padrão porque o Synthea exige Java e produz
centenas de MB, inviável em CI — e porque um gerador determinístico permite testes de recuperação exata.

Uma tabela adicional, `lab_orders`, modela **exames pendentes** — informação exigida pelo enunciado e
ausente do export padrão do Synthea.

In [ ]:
from medflow_ai.database.ingest import build_synthetic_database
from medflow_ai.database.schema import SCHEMA_SQL

contagens = build_synthetic_database(n_patients=40)
print(contagens)
print(SCHEMA_SQL[:1200])

## 2. Consultas ao prontuário

In [ ]:
from medflow_ai.database.repository import PatientRepository

repo = PatientRepository()
PACIENTE = "P-DEMO-0001"

print("Condições ativas   :", repo.conditions(PACIENTE))
print("\nExames recentes  :", repo.latest_observations(PACIENTE, limit=5))
print("\nMedicamentos      :", repo.medications(PACIENTE))
print("\nExames pendentes  :", repo.pending_exams(PACIENTE))
print("\nÚltimos encontros :", repo.encounters(PACIENTE, limit=2))

In [ ]:
# Tendência de um exame ao longo do tempo — usada para contextualizar a resposta
import pandas as pd
historico = pd.DataFrame(repo.observation_history(PACIENTE, "TSH"))
historico

## 3. A LLM recebe identificadores diretos? (evidência de minimização)

**Pergunta.** O que exatamente atravessa a fronteira entre o banco e o prompt?

In [ ]:
bruto = repo.raw_record(PACIENTE)
print("=== REGISTRO BRUTO NO BANCO (nunca sai daqui) ===")
for chave, valor in bruto.items():
    print(f"  {chave:12s}: {valor}")

In [ ]:
contexto = repo.build_context(PACIENTE)
bloco = contexto.to_prompt_block()
print("=== O QUE A LLM EFETIVAMENTE RECEBE ===")
print(bloco)

In [ ]:
# Verificação explícita: nenhum identificador direto atravessou
vazou = [campo for campo in ("first", "last", "cpf", "cns", "email", "phone", "address", "birthdate")
         if str(bruto[campo]) and str(bruto[campo]) in bloco]
print("Campos identificadores presentes no prompt:", vazou or "NENHUM ✅")

from medflow_ai.data.anonymization import contains_pii
print("Detector de PII acusa algo no prompt?", contains_pii(bloco))

**Interpretação.** O nome, CPF, CNS, telefone, e-mail e endereço permanecem no banco e **não**
chegam ao prompt. A data de nascimento é substituída por faixa etária, e o `patient_id` por um
pseudônimo derivado com salt. O que sobe ao modelo é o mínimo necessário para responder.

## 4. Minimização sob demanda

O contexto pode ser reduzido ainda mais conforme a pergunta — se o médico só quer saber de pendências,
não há razão para enviar histórico medicamentoso.

In [ ]:
apenas_pendencias = repo.build_context(PACIENTE, include=["pending_exams"])
print(apenas_pendencias.to_prompt_block())
print("\nCondições enviadas:", apenas_pendencias.conditions)

## 5. Ferramentas LangChain que expõem essa base ao grafo

In [ ]:
from medflow_ai.graph.tools import MEDFLOW_TOOLS, consultar_prontuario, verificar_alertas_clinicos

for ferramenta in MEDFLOW_TOOLS:
    print(f"- {ferramenta.name}: {ferramenta.description.splitlines()[0]}")

print("\nAlertas automáticos do paciente demo:")
for alerta in verificar_alertas_clinicos.invoke({"patient_id": PACIENTE}):
    print(f"  [{alerta['severidade'].upper()}] {alerta['mensagem']} (fonte: {alerta['fonte']})")

## 6. Recuperação exata é verificável?

**Método.** O gabarito é derivado do próprio banco: para cada exame, o valor mais recente conhecido é
comparado com o valor devolvido pelo repositório.

In [ ]:
from medflow_ai.evaluation.database_eval import build_cases, evaluate_database

relatorio = evaluate_database(repo, build_cases(limit=30))
print(f"recuperação exata: {relatorio.exact_match:.3f} em {relatorio.n_cases} casos")
print(f"contexto livre de identificadores diretos: {relatorio.context_leak_free}")
print("falhas:", relatorio.falhas or "nenhuma")

**Interpretação.** Diferente do RAG, aqui não há ambiguidade: ou o valor bate, ou não bate. Este é
o teste que distingue "a LLM disse algo plausível" de "o dado do paciente foi realmente usado".

**Limitação.** Os pacientes são sintéticos e clinicamente simplificados; um prontuário real traz texto
livre, evoluções, exames de imagem e inconsistências que este modelo não representa.